In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import os
import json
import pathlib
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get the current directory (using pathlib instead of __file__ which doesn't work in notebooks)
current_dir = str(pathlib.Path().absolute())

# Format md files to be more readable
endpoint = os.getenv("endpoint")
openai_api_key = os.getenv("openai_api_key")

gpt4omini_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-large"

# Initialize Azure OpenAI Service client with key-based authentication    
client = AzureOpenAI(
    azure_endpoint=endpoint,  
    api_key=openai_api_key,
    api_version="2025-01-01-preview",
)

def cosine_similarity(vec1, vec2):
    """
    Calculate the cosine similarity between two vectors.

    Args:
        vec1 (array-like): First vector.
        vec2 (array-like): Second vector.

    Returns:
        float: Cosine similarity score between -1 and 1.

    Raises:
        ValueError: If the vectors are not the same shape or if one vector is zero.
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    if vec1.shape != vec2.shape:
        raise ValueError("Vectors must have the same dimensions.")
    
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        raise ValueError("One of the vectors is zero and cannot be normalized.")
    
    return np.dot(vec1, vec2) / (norm1 * norm2)


In [2]:
import re
clean_mobile1 = re.sub(r'[^\d]', '', str("+639211234571"))
print(clean_mobile1)

639211234571


# <span style="color:yellow;">Vectorize Details</span>


In [2]:
import os
import json
import pathlib

current_dir = str(pathlib.Path().absolute())
db_path = os.path.join(current_dir, "local_db.json")

# Open the local database file
with open(db_path, 'r', encoding='utf-8') as f:
    tiketsdb = json.load(f)

print(f"Loaded {len(tiketsdb)} tickets from the local database.")

# Loop through the tickets
for ticket in tiketsdb:
    print(f"details: {ticket['details']}")
    embeddings = client.embeddings.create(
        input=ticket['details'],
        model=embedding_model
    )

    ticket['details_vector'] = embeddings.data[0].embedding
    # break

# Save the updated tickets back to the local database
with open(db_path, 'w', encoding='utf-8') as f:
    json.dump(tiketsdb, f, indent=4)
    print(f"Updated {len(tiketsdb)} tickets with embeddings and saved to {db_path}.")


Loaded 10 tickets from the local database.
details: I can't log into my account since yesterday.
details: I made a payment but it's not reflecting.
details: The app crashes every time I open it.
details: Password reset link is not working.
details: My account is locked after multiple failed attempts.
details: Please add a dark mode option.
details: Reports are missing data fields.
details: Clicking the save button does not trigger any action.
details: The edit option is visible but unresponsive when clicked.
details: Profile update button is not responding.
Updated 10 tickets with embeddings and saved to c:\Users\robertrita\Workspace\CODES\_MVP\xcash\local_db.json.


# <span style="color:yellow;">LAB-2A: 🧠Semantic Search + Chat Completion</span>

In [ ]:
#Prepare the chat prompt
system_retrieval = """
You are a helpful BSP AI Assistant. Follow these rules exactly:

1. PURPOSE
   - Use the supplied DOCUMENT_CHUNK to answer user questions.
   - Allow simple greetings and light chitchat.
   - If the user's question is outside the scope of DOCUMENT_CHUNK, respond with a polite refusal.

2. INPUT FORMAT
   The model always receives two inputs in this order:
   a) DOCUMENT_CHUNK: A block of text containing the relevant information.
   b) USER_QUERY: The user's message or question.

3. BEHAVIOR
   a) Greetings & Chitchat
      - If USER_QUERY is a greeting (e.g. “hi”, “hello”, “good morning”) or simple chitchat (“how are you?”, “what's up?”), respond with a friendly greeting or brief chitchat. Do not reference DOCUMENT_CHUNK.
   b) On-Topic Questions
      - If USER_QUERY asks about a fact or detail that is directly supported by DOCUMENT_CHUNK, answer accurately using only information from DOCUMENT_CHUNK.
      - Cite the relevant passage or phrase when possible: “According to the document: ‹…›”.
   c) Off-Topic or Irrelevant Questions
      - If USER_QUERY cannot be answered from DOCUMENT_CHUNK, reply:
        “I'm sorry, but I don't have information on that. Please ask something related to the document.”
      - Do NOT attempt to hallucinate or introduce outside knowledge.

4. RESPONSE FORMAT
   - Keep answers concise (2-4 sentences).
   - Use neutral, professional tone.
   - If refusing, use exactly: “I'm sorry, but I don't have information on that. Please ask something related to the document.”

5. EXAMPLES

Example 1 - Greeting  
DOCUMENT_CHUNK: “...”  
USER_QUERY: “Hey there!”  
→ “Hello! How can I help you with the document today?”

Example 2 - On-Topic  
DOCUMENT_CHUNK: “The Eiffel Tower is 300 meters tall.”  
USER_QUERY: “How tall is the Eiffel Tower?”  
→ “According to the document, the Eiffel Tower is 300 meters tall.”

Example 3 - Off-Topic  
DOCUMENT_CHUNK: “...”  
USER_QUERY: “What's the weather today?”  
→ “I'm sorry, but I don't have information on that. Please ask something related to the document.”
"""

user_retrieval = """
DOCUMENT_CHUNK: {{DOCUMENT}}
USER_QUERY: {{QUERY}}
"""

# - Return the last query and last intent with complete context in English language. without any additional information or context.
system_rewrite = """
You are an expert copywriter.
- Given the following chat history, precisely extract the last query and last intent made by the user.
- Return the last query and last intent in English language. without any additional information or context.
- The output should contain the following details:
1. query: The last query made by the user.
2. intent: The last intent made by the user.
"""

user_rewrite = """
Chat History:
{{user_input}}
"""


# Function that rewrites the user input
def rewrite_query(user_input):
    # Create a new system prompt for each file by replacing the placeholder
    user_prompt = user_rewrite.replace("{{user_input}}", user_input)
    messages = [
        {"role": "system", "content": system_rewrite},
        {"role": "user", "content": user_prompt},
    ]

    # Generate the completion  
    completion = client.chat.completions.create(  
        model=gpt4omini_model,
        messages=messages,
        temperature=0,
        top_p=1,
    )

    return completion.choices[0].message.content


# Get a list of all Markdown files in the md folder
vector_files = [f for f in os.listdir(faq_dir) if f.lower().endswith('.json')]

def vector_search(query):
    # Create the vector for the query
    query_vector = client.embeddings.create(
        input=query,
        model=embedding_model
    )
    results = []

    # Search for the most relevant documents using the vector
    for vector_file in vector_files:
        # Create full path for input file
        vector_path = os.path.join(faq_dir, vector_file)
        # print(f"Processing: {vector_path}")
        
        with open(vector_path, 'r', encoding='utf-8') as f:
            vectors = json.load(f)

        # Loop through the vectors and check for similarity
        for vector in vectors:
            # Check if the vector is similar to the query vector
            similarity = cosine_similarity(vector['vector'], query_vector.data[0].embedding)
            # print(f"Similarity: {similarity} / {vector['chunkId']}")
            
            # If the similarity is above a certain threshold, add it to the results
            if float(similarity) > 0.5:
                results.append({
                    'content': f"{vector['topic']}\n{vector['content']}",
                    'similarity': round(similarity, 2),
                })

    return results


def chat_with_pdf(user_query):
    try:
        print(f"User Query: {user_query}")
        # user_query = rewrite_query(user_query)
        # print(f"Rewrites: {user_query}")

        search = vector_search(user_query)
        # print(f"Search results: {json.dumps(search, indent=4)}")

        # Sort the data by similarity in descending order
        top5 = sorted(search, key=lambda x: x['similarity'], reverse=True)[:5]
        print("Top Doc Chunks: ", top5)

        # join with spaces (or use "\n" for newlines)
        documents = "\n\n".join(item['content'] for item in top5)

        # Create a new system prompt for each file by replacing the placeholder
        user_prompt = user_retrieval.replace("{{DOCUMENT}}", documents).replace("{{QUERY}}", user_query)
        messages = [
            {"role": "system", "content": system_retrieval},
            {"role": "user", "content": user_prompt},
        ]

        # Generate the completion  
        completion = client.chat.completions.create(  
            model=gpt4omini_model,
            messages=messages,
            temperature=0,
            top_p=1,
            response_format={ "type": "text" },
        )

        response = completion.choices[0].message.content
        print(f"Response: {response}")
        results = {
            "query": user_query,
            "response": response,
            "documents": documents,
        }

        # Save the structured data to a JSON file
        output_path = os.path.join(current_dir, 'results.json')

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=4)
        print(f"Results saved to: {output_path}")

    except Exception as e:
        print(f"Error occurred: {e}")


chat_with_pdf("ano ang NPSA?")

User Query: ano ang NPSA?
Top Doc Chunks:  [{'content': 'What is the National Payment Systems Act (NPSA)?\nThe NPSA is a landmark legislation that supports the performance by the Bangko Sentral of its mandate relating to the third pillar of central banking – the maintenance of a safe, efficient, and reliable payment and settlement systems.', 'similarity': np.float64(0.68)}, {'content': 'What are the objectives of the NPSA?\nAs the first comprehensive legal and regulatory framework governing payment systems in the Philippines, the NPSA supports the twin objectives of maintaining safe, secure, efficient and reliable operations of payment systems that is necessary to control systemic risk and of providing an environment conducive to the sustainable growth of the economy.', 'similarity': np.float64(0.66)}, {'content': 'What is BSP Circular No. 1049 about? What is its objective in relation to the first phase of implementation of the NPSA?\nCircular No. 1049 provides the rules and regulation

# <span style="color:yellow;">LAB-2B: Evaluate Groundedness for RAG</span>
https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/evaluation/azure-ai-evaluation/azure/ai/evaluation/_evaluators/_groundedness/groundedness_with_query.prompty

In [26]:
system_eval = """
# Instruction
## Goal
### You are an expert in evaluating the quality of a RESPONSE from an intelligent system based on provided definition and data. Your goal will involve answering the questions below using the information provided.
- **Definition**: You are given a definition of the communication trait that is being evaluated to help guide your Score.
- **Data**: Your input data include CONTEXT, QUERY, and RESPONSE.
- **Tasks**: To complete your evaluation you will be asked to evaluate the Data in different ways.
"""

user_eval = """
# Definition
**Groundedness** refers to how well an answer is anchored in the provided context, evaluating its relevance, accuracy, and completeness based exclusively on that context. It assesses the extent to which the answer directly and fully addresses the question without introducing unrelated or incorrect information. The scale ranges from 1 to 5, with higher numbers indicating greater groundedness.

# Ratings
## [Groundedness: 1] (Completely Unrelated Response)
**Definition:** An answer that does not relate to the question or the context in any way. It fails to address the topic, provides irrelevant information, or introduces completely unrelated subjects.

**Examples:**
  **Context:** The company's annual meeting will be held next Thursday.
  **Query:** When is the company's annual meeting?
  **Response:** I enjoy hiking in the mountains during summer.

  **Context:** The new policy aims to reduce carbon emissions by 20% over the next five years.
  **Query:** What is the goal of the new policy?
  **Response:** My favorite color is blue.

## [Groundedness: 2] (Related Topic but Does Not Respond to the Query)
**Definition:** An answer that relates to the general topic of the context but does not answer the specific question asked. It may mention concepts from the context but fails to provide a direct or relevant response.

**Examples:**
  **Context:** The museum will exhibit modern art pieces from various local artists.
  **Query:** What kind of art will be exhibited at the museum?
  **Response:** Museums are important cultural institutions.

  **Context:** The new software update improves battery life and performance.
  **Query:** What does the new software update improve?
  **Response:** Software updates can sometimes fix bugs.

## [Groundedness: 3] (Attempts to Respond but Contains Incorrect Information)
**Definition:** An answer that attempts to respond to the question but includes incorrect information not supported by the context. It may misstate facts, misinterpret the context, or provide erroneous details.

**Examples:**
  **Context:** The festival starts on June 5th and features international musicians.
  **Query:** When does the festival start?
  **Response:** The festival starts on July 5th and features local artists.

  **Context:** The recipe requires two eggs and one cup of milk.
  **Query:** How many eggs are needed for the recipe?
  **Response:** You need three eggs for the recipe.

## [Groundedness: 4] (Partially Correct Response)
**Definition:** An answer that provides a correct response to the question but is incomplete or lacks specific details mentioned in the context. It captures some of the necessary information but omits key elements needed for a full understanding.

**Examples:**
  **Context:** The bookstore offers a 15% discount to students and a 10% discount to senior citizens.
  **Query:** What discount does the bookstore offer to students?
  **Response:** Students get a discount at the bookstore.

  **Context:** The company's headquarters are located in Berlin, Germany.
  **Query:** Where are the company's headquarters?
  **Response:** The company's headquarters are in Germany.

## [Groundedness: 5] (Fully Correct and Complete Response)
**Definition:** An answer that thoroughly and accurately responds to the question, including all relevant details from the context. It directly addresses the question with precise information, demonstrating complete understanding without adding extraneous information.

**Examples:**
  **Context:** The author released her latest novel, 'The Silent Echo', on September 1st.
  **Query:** When was 'The Silent Echo' released?
  **Response:** 'The Silent Echo' was released on September 1st.

  **Context:** Participants must register by May 31st to be eligible for early bird pricing.
  **Query:** By what date must participants register to receive early bird pricing?
  **Response:** Participants must register by May 31st to receive early bird pricing.


# Data
CONTEXT: {{context}}
QUERY: {{query}}
RESPONSE: {{response}}


# Tasks
## Please provide your assessment Score for the previous RESPONSE in relation to the CONTEXT and QUERY based on the Definitions above. Your output should include the following information:
- **ThoughtChain**: To improve the reasoning process, think step by step and include a step-by-step explanation of your thought process as you analyze the data based on the definitions. Keep it brief and start your ThoughtChain with "Let's think step by step:".
- **Explanation**: a very short explanation of why you think the input Data should get that Score.
- **Score**: based on your previous analysis, provide your Score. The Score you give MUST be a integer score (i.e., "1", "2"...) based on the levels of the definitions.

## Please provide your answers in JSON object with the following structure:
{
    "thought_chain": "<chain of thoughts>",
    "explanation": "<your explanation>",
    "score": "<your Score>"
}
"""

def check_evals(user_queries):
  # Loop through the contents
  for user_query in user_queries:
    chat_with_pdf(user_query)

    results_path = os.path.join(current_dir, 'results.json')

    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
        # print(f"Results: {results}")

    response, documents = results['response'], results['documents']
    # print(f"Response: {response}")
    # print(f"Documents: {documents}")

    user_prompt = user_eval.replace("{{query}}", user_query).replace("{{context}}", documents).replace("{{response}}", response)
    messages = [
        {"role": "system", "content": system_eval},
        {"role": "user", "content": user_prompt},
    ]

    completion = client.chat.completions.create(
        model=gpt4omini_model,
        messages=messages,
        temperature=0,
        top_p=1,
        response_format={ "type": "json_object" },
    )

    evals = completion.choices[0].message.content
    print(evals)


user_queries = [
  #  "hello",
  #   "what is NPSA?",
    "what is BSP?",
]

check_evals(user_queries)

User Query: what is BSP?
Top Doc Chunks:  []
Response: I'm sorry, but I don't have information on that. Please ask something related to the document.
Results saved to: c:\Users\robertrita\Workspace\CODES\_Mine\Chat-With-Data\notebooks\results.json
{
    "thought_chain": "Let's think step by step: The context does not provide any information about what 'BSP' is, nor does it mention any related topics. The query specifically asks for information about 'BSP', which is not addressed in the response. The response states a lack of information and suggests asking something related to the document, which does not help in answering the query. Therefore, the response does not relate to the query or context at all.",
    "explanation": "The response does not provide any relevant information about 'BSP' and fails to address the query, making it completely unrelated to the context.",
    "score": "1"
}
